In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path("ecommerce.sqlite")
STAGING = Path("spark_input")
STAGING.mkdir(exist_ok=True)

TABLES = [
    "geolocation", "order_items", "products",
    "sellers", "product_category_name_translation"
]

with sqlite3.connect(DB_PATH) as con:
    for table in TABLES:
        out = STAGING / f"{table}.csv"
        first = True
        total = 0
        for chunk in pd.read_sql_query(
            f"SELECT * FROM {table}", con, chunksize=100_000
        ):
            chunk.to_csv(
                out, index=False,
                mode="w" if first else "a",
                header=first, encoding="utf-8"
            )
            total += len(chunk)
            first = False
        print(f"{table:40s} {total:>9d} rows")

geolocation                                1000163 rows
order_items                                 112650 rows
products                                     32951 rows
sellers                                       3095 rows
product_category_name_translation               71 rows


In [2]:
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel

spark = (
    SparkSession.builder
        .master("local[*]")
        .appName("Practice5_SparkExecution")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.adaptive.enabled", "false")
        .config("spark.sql.autoBroadcastJoinThreshold", "-1")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Master:", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions:",
spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Spark UI:", spark.sparkContext.uiWebUrl)

C:\Additional\VirtualMachines\prac4\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Master: local[*]
Default parallelism: 12
Shuffle partitions: 8
AQE: false
Spark UI: http://MSI:4040


In [3]:
PARQUET = Path("spark_input_parquet")
PARQUET.mkdir(exist_ok=True)

for name in TABLES:
    df = (
        spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(str(STAGING / f"{name}.csv"))
        )
    df.write.mode("overwrite").parquet(str(PARQUET / name))
    
geo = spark.read.parquet(str(PARQUET / "geolocation"))
items = spark.read.parquet(str(PARQUET / "order_items"))
products = spark.read.parquet(str(PARQUET / "products"))
sellers = spark.read.parquet(str(PARQUET / "sellers"))
translations = spark.read.parquet(
    str(PARQUET / "product_category_name_translation")
)

print(geo.count(), items.count(), products.count(), sellers.count())

1000163 112650 32951 3095


In [4]:
sc = spark.sparkContext
sc.setJobGroup(
    "P5_BASELINE",
    "Практика 5: базовая проверка Spark UI",
    interruptOnCancel=True
)

_ = items.count()

In [5]:
sc.setJobGroup("P5_LAZY", "Lazy evaluation")

geo_prepared = (
    geo
        .select(
            "geolocation_zip_code_prefix",
            "geolocation_state",
            "geolocation_lat",
            "geolocation_lng"
        )
        .filter(F.col("geolocation_state").isNotNull())
        .withColumn("abs_lat", F.abs(F.col("geolocation_lat")))
)

print(type(geo_prepared).__name__)

DataFrame


In [6]:
from time import perf_counter

t0 = perf_counter()
n = geo_prepared.count()
elapsed = perf_counter() - t0

print("Rows:", n)
print(f"Elapsed: {elapsed:.3f} s")

Rows: 1000163
Elapsed: 0.703 s


In [7]:
print("Input partitions:", geo.rdd.getNumPartitions())
print("Default parallelism:", sc.defaultParallelism)

partition_sizes = (
    geo
        .select(F.spark_partition_id().alias("pid"))
        .groupBy("pid")
        .count()
        .orderBy("pid")
    )
partition_sizes.show(100, truncate=False)

Input partitions: 12
Default parallelism: 12
+---+-----+
|pid|count|
+---+-----+
|0  |90353|
|1  |88373|
|2  |88104|
|3  |87668|
|4  |86154|
|5  |88355|
|6  |87983|
|7  |86853|
|8  |86841|
|9  |91182|
|10 |92908|
|11 |25389|
+---+-----+



In [8]:
state_counts = (
    geo
        .groupBy("geolocation_state")
        .count()
        .orderBy(F.desc("count"))
    )
state_counts.show(5, truncate=False)

+-----------------+------+
|geolocation_state|count |
+-----------------+------+
|SP               |404268|
|MG               |126336|
|RJ               |121169|
|RS               |61851 |
|PR               |57859 |
+-----------------+------+
only showing top 5 rows


In [9]:
skewed16 = geo.repartition(16, "geolocation_state")
repart4 = skewed16.repartition(4)
coal4 = skewed16.coalesce(4)

print("skewed16:", skewed16.rdd.getNumPartitions())
print("repart4:", repart4.rdd.getNumPartitions())
print("coal4:", coal4.rdd.getNumPartitions())

skewed16: 16
repart4: 4
coal4: 4


In [10]:
print("=== repartition(4) ===")
repart4.explain("formatted")

print("=== coalesce(4) ===")
coal4.explain("formatted")

=== repartition(4) ===
== Physical Plan ==
Exchange (3)
+- * ColumnarToRow (2)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [6]: [geolocation_id#141, geolocation_zip_code_prefix#142, geolocation_lat#143, geolocation_lng#144, geolocation_city#145, geolocation_state#146]
Batched: true
Location: InMemoryFileIndex [file:/C:/Additional/VirtualMachines/prac4/spark_input_parquet/geolocation]
ReadSchema: struct<geolocation_id:int,geolocation_zip_code_prefix:int,geolocation_lat:double,geolocation_lng:double,geolocation_city:string,geolocation_state:string>

(2) ColumnarToRow [codegen id : 1]
Input [6]: [geolocation_id#141, geolocation_zip_code_prefix#142, geolocation_lat#143, geolocation_lng#144, geolocation_city#145, geolocation_state#146]

(3) Exchange
Input [6]: [geolocation_id#141, geolocation_zip_code_prefix#142, geolocation_lat#143, geolocation_lng#144, geolocation_city#145, geolocation_state#146]
Arguments: RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=466]


=== coalesce

In [11]:
def partition_profile(df):
    return (
        df
            .select(F.spark_partition_id().alias("pid"))
            .groupBy("pid")
            .count()
            .orderBy("pid")
        )

print("repartition(4)")
partition_profile(repart4).show()

print("coalesce(4)")
partition_profile(coal4).show()

repartition(4)
+---+------+
|pid| count|
+---+------+
|  0|250040|
|  1|250041|
|  2|250041|
|  3|250041|
+---+------+

coalesce(4)
+---+------+
|pid| count|
+---+------+
|  0|195064|
|  1|626485|
|  2| 90152|
|  3| 88462|
+---+------+



In [12]:
sc.setJobGroup("P5_GROUPBY", "Shuffle: groupBy seller_id")

seller_stats = (
    items
        .groupBy("seller_id")
        .agg(
            F.count("*").alias("items"),
            F.round(F.sum("price"), 2).alias("revenue"),
            F.round(F.avg("price"), 2).alias("avg_price")
            )
        .orderBy(F.desc("revenue"))
    )

seller_stats.show(5, truncate=False)

+--------------------------------+-----+---------+---------+
|seller_id                       |items|revenue  |avg_price|
+--------------------------------+-----+---------+---------+
|4869f7a5dfa277a7dca6462dcf3b52b2|1156 |229472.63|198.51   |
|53243585a1d6dc2643021fd1853d8905|410  |222776.05|543.36   |
|4a3ca9315b744ce9f8e9374361493884|1987 |200472.92|100.89   |
|fa1c13f2614d7b5c4749cbc52fecda94|586  |194042.03|331.13   |
|7c67e1448b00f6e969d365cea6b010ab|1364 |187923.89|137.77   |
+--------------------------------+-----+---------+---------+
only showing top 5 rows


In [13]:
seller_stats.explain("formatted")

== Physical Plan ==
* Sort (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- Exchange (4)
         +- * HashAggregate (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [2]: [seller_id#150, price#152]
Batched: true
Location: InMemoryFileIndex [file:/C:/Additional/VirtualMachines/prac4/spark_input_parquet/order_items]
ReadSchema: struct<seller_id:string,price:double>

(2) ColumnarToRow [codegen id : 1]
Input [2]: [seller_id#150, price#152]

(3) HashAggregate [codegen id : 1]
Input [2]: [seller_id#150, price#152]
Keys [1]: [seller_id#150]
Functions [3]: [partial_count(1), partial_sum(price#152), partial_avg(price#152)]
Aggregate Attributes [4]: [count#307L, sum#308, sum#309, count#310L]
Results [5]: [seller_id#150, count#311L, sum#312, sum#313, count#314L]

(4) Exchange
Input [5]: [seller_id#150, count#311L, sum#312, sum#313, count#314L]
Arguments: hashpartitioning(seller_id#150, 8), ENSURE_REQUIREMENTS, [plan_id=716]

(5) HashAggre

In [14]:
plan_demo = (
    items
        .filter(F.col("price") >= 100)
        .select("seller_id", "price", "freight_value")
        .withColumn("gross", F.col("price") + F.col("freight_value"))
        .groupBy("seller_id")
        .agg(F.round(F.sum("gross"), 2).alias("gross_sum"))
    )

plan_demo.explain("formatted")

== Physical Plan ==
* HashAggregate (7)
+- Exchange (6)
   +- * HashAggregate (5)
      +- * Project (4)
         +- * Filter (3)
            +- * ColumnarToRow (2)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [3]: [seller_id#150, price#152, freight_value#153]
Batched: true
Location: InMemoryFileIndex [file:/C:/Additional/VirtualMachines/prac4/spark_input_parquet/order_items]
PushedFilters: [IsNotNull(price), GreaterThanOrEqual(price,100.0)]
ReadSchema: struct<seller_id:string,price:double,freight_value:double>

(2) ColumnarToRow [codegen id : 1]
Input [3]: [seller_id#150, price#152, freight_value#153]

(3) Filter [codegen id : 1]
Input [3]: [seller_id#150, price#152, freight_value#153]
Condition : (isnotnull(price#152) AND (price#152 >= 100.0))

(4) Project [codegen id : 1]
Output [2]: [seller_id#150, (price#152 + freight_value#153) AS gross#336]
Input [3]: [seller_id#150, price#152, freight_value#153]

(5) HashAggregate [codegen id : 1]
Input [2]: [seller_id#150, gr

In [15]:
sc.setJobGroup("P5_SMJ", "Sort Merge Join: items x products")

smj = (
    items.hint("merge")
        .join(products.hint("merge"), on="product_id", how="inner")
    )

smj.explain("formatted")
print("Rows:", smj.count())

== Physical Plan ==
* Project (12)
+- * SortMergeJoin Inner (11)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Filter (3)
   :        +- * ColumnarToRow (2)
   :           +- Scan parquet  (1)
   +- * Sort (10)
      +- Exchange (9)
         +- * Filter (8)
            +- * ColumnarToRow (7)
               +- Scan parquet  (6)


(1) Scan parquet 
Output [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]
Batched: true
Location: InMemoryFileIndex [file:/C:/Additional/VirtualMachines/prac4/spark_input_parquet/order_items]
PushedFilters: [IsNotNull(product_id)]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:double,freight_value:double>

(2) ColumnarToRow [codegen id : 1]
Input [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]

(3) Filter [codegen id : 1]
Input 

In [16]:
sc.setJobGroup("P5_BHJ", "Broadcast Hash Join: items x products")

bhj = items.join(
    F.broadcast(products),
    on="product_id",
    how="inner"
    )

bhj.explain("formatted")
print("Rows:", bhj.count())

== Physical Plan ==
* Project (9)
+- * BroadcastHashJoin Inner BuildRight (8)
   :- * Filter (3)
   :  +- * ColumnarToRow (2)
   :     +- Scan parquet  (1)
   +- BroadcastExchange (7)
      +- * Filter (6)
         +- * ColumnarToRow (5)
            +- Scan parquet  (4)


(1) Scan parquet 
Output [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]
Batched: true
Location: InMemoryFileIndex [file:/C:/Additional/VirtualMachines/prac4/spark_input_parquet/order_items]
PushedFilters: [IsNotNull(product_id)]
ReadSchema: struct<order_id:string,order_item_id:int,product_id:string,seller_id:string,shipping_limit_date:timestamp,price:double,freight_value:double>

(2) ColumnarToRow [codegen id : 2]
Input [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#150, shipping_limit_date#151, price#152, freight_value#153]

(3) Filter [codegen id : 2]
Input [7]: [order_id#147, order_item_id#148, product_id#149, seller_id#

In [17]:
spark.catalog.clearCache()

products_small = products.select(
    "product_id", "product_category_name"
)
sellers_small = sellers.select(
    "seller_id", "seller_state"
)

joined = (
    items
        .join(F.broadcast(products_small), "product_id", "left")
        .join(F.broadcast(sellers_small), "seller_id", "left")
        .withColumn(
            "gross_value",
            F.col("price") + F.col("freight_value")
            )
    )

def action_category(df):
    return (
        df.groupBy("product_category_name")
            .agg(F.sum("gross_value").alias("gross"))
            .agg(F.sum("gross").alias("checksum"))
            .first()[0]
        )
    
def action_state(df):
    return (
        df.groupBy("seller_state")
            .agg(F.sum("gross_value").alias("gross"))
            .agg(F.sum("gross").alias("checksum"))
            .first()[0]
        )

In [18]:
sc.setJobGroup("P5_NOCACHE", "Два действия без сохранения")
t0 = perf_counter()
_ = action_category(joined)
t1 = perf_counter()
_ = action_state(joined)
t2 = perf_counter()
print(f"Action 1: {t1 - t0:.3f} s")
print(f"Action 2: {t2 - t1:.3f} s")
print("is_cached:", joined.is_cached)

Action 1: 0.795 s
Action 2: 0.592 s
is_cached: False


In [19]:
cached = joined.persist(StorageLevel.MEMORY_AND_DISK)
print("После persist:", cached.is_cached)

# Materialization: первый Action заполняет кэш
t0 = perf_counter()
materialized_rows = cached.count()
t1 = perf_counter()

# Повторные Actions используют материализованный DataFrame
_ = action_category(cached)
t2 = perf_counter()
_ = action_state(cached)
t3 = perf_counter()

print("Rows:", materialized_rows)
print(f"Materialization: {t1 - t0:.3f} s")
print(f"Cached action 1: {t2 - t1:.3f} s")
print(f"Cached action 2: {t3 - t2:.3f} s")

cached.unpersist(blocking=True)
print("После unpersist:", cached.is_cached)

После persist: True
Rows: 112650
Materialization: 1.010 s
Cached action 1: 0.537 s
Cached action 2: 0.293 s
После unpersist: False


In [20]:
from statistics import median

N = 5_000_000
P_VALUES = [1, 2, 4, 8, 16, 32]
spark.conf.set("spark.sql.shuffle.partitions", "8")

def run_benchmark(p, repeats=3):
    times = []
    checksum = None
    for _ in range(repeats):
        df = (
            spark.range(0, N, 1, numPartitions=p)
                .withColumn("key", (F.col("id") % 1000).cast("int"))
                .withColumn("value", F.sqrt(F.col("id") + 1.0))
        )
        t0 = perf_counter()
        checksum = (
            df.groupBy("key")
                .agg(F.sum("value").alias("s"))
                .agg(F.sum("s").alias("checksum"))
                .first()[0]
        )
        times.append(perf_counter() - t0)
    return times, median(times), checksum

# Прогрев
_ = run_benchmark(4, repeats=1)

results = []
for p in P_VALUES:
    times, med, checksum = run_benchmark(p, repeats=3)
    results.append((p, *times, med))
    print(p, times, med, checksum)

1 [0.28362149999884423, 0.16547630000422942, 0.2599037000036333] 0.2599037000036333 7453561042.825421
2 [0.6486975000007078, 0.7018434000056004, 0.34193400000367546] 0.6486975000007078 7453561042.82542
4 [0.4864335000020219, 0.6116164999984903, 0.4211278000002494] 0.4864335000020219 7453561042.825421
8 [0.6016802000012831, 0.6161572000055457, 0.41333619999932125] 0.6016802000012831 7453561042.825422
16 [1.155653000001621, 0.9461009000006015, 0.8725406000012299] 0.9461009000006015 7453561042.82542
32 [1.828165800005081, 1.8444747000030475, 1.277529200000572] 1.828165800005081 7453561042.825421


In [21]:
geo8 = geo.repartition(8)
Q_VALUES = [2, 4, 8, 16, 32]

for q in Q_VALUES:
    spark.conf.set("spark.sql.shuffle.partitions", str(q))
    t0 = perf_counter()
    rows = (
        geo8.groupBy("geolocation_state")
            .count()
            .count()
        )
    elapsed = perf_counter() - t0
    print(f"q={q:2d}; groups={rows}; time={elapsed:.3f}s")

# Возвращаем базовую настройку
spark.conf.set("spark.sql.shuffle.partitions", "8")

q= 2; groups=27; time=1.256s
q= 4; groups=27; time=0.971s
q= 8; groups=27; time=0.933s
q=16; groups=27; time=1.220s
q=32; groups=27; time=1.469s


In [22]:
OUT_SMALL = Path("small_files_experiment")
OUT_SMALL.mkdir(exist_ok=True)

def write_and_measure(p):
    path = OUT_SMALL / f"p_{p}"
    (
        items.repartition(p)
            .write.mode("overwrite")
            .parquet(str(path))
        )
    files = list(path.glob("part-*.parquet"))
    sizes = [f.stat().st_size for f in files]
    total_mib = sum(sizes) / 1024**2
    avg_kib = (sum(sizes) / len(sizes) / 1024) if sizes else 0
    return len(files), total_mib, avg_kib
    
for p in [1, 4, 16, 64]:
    print(p, write_and_measure(p))

1 (1, 6.456236839294434, 6611.1865234375)
4 (4, 7.432483673095703, 1902.7158203125)
16 (16, 8.732264518737793, 558.8649291992188)
64 (64, 10.285961151123047, 164.57537841796875)


In [23]:
single = items.coalesce(1)
print("Partitions before write:", single.rdd.getNumPartitions())

# Выполнять только как учебный пример на небольшом order_items
(single.write.mode("overwrite")
    .parquet(str(OUT_SMALL / "coalesce_1")))

Partitions before write: 1


In [24]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "32")

adaptive = (
    geo
        .groupBy("geolocation_state")
        .count()
)

adaptive.explain("formatted")
adaptive.collect()

print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))

== Physical Plan ==
AdaptiveSparkPlan (5)
+- HashAggregate (4)
   +- Exchange (3)
      +- HashAggregate (2)
         +- Scan parquet  (1)


(1) Scan parquet 
Output [1]: [geolocation_state#146]
Batched: true
Location: InMemoryFileIndex [file:/C:/Additional/VirtualMachines/prac4/spark_input_parquet/geolocation]
ReadSchema: struct<geolocation_state:string>

(2) HashAggregate
Input [1]: [geolocation_state#146]
Keys [1]: [geolocation_state#146]
Functions [1]: [partial_count(1)]
Aggregate Attributes [1]: [count#1743L]
Results [2]: [geolocation_state#146, count#1744L]

(3) Exchange
Input [2]: [geolocation_state#146, count#1744L]
Arguments: hashpartitioning(geolocation_state#146, 32), ENSURE_REQUIREMENTS, [plan_id=3064]

(4) HashAggregate
Input [2]: [geolocation_state#146, count#1744L]
Keys [1]: [geolocation_state#146]
Functions [1]: [count(1)]
Aggregate Attributes [1]: [count(1)#1742L]
Results [2]: [geolocation_state#146, count(1)#1742L AS count#1735L]

(5) AdaptiveSparkPlan
Output [2]: [ge

In [25]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "8")

In [26]:
try:
    spark.stop()
    print("SparkSession остановлена")
except Exception as e:
    print("Spark уже остановлена:", e)

try:
    spark.catalog.clearCache()
except Exception:
    pass

import time
time.sleep(2)
try:
    print("UI URL после stop:", spark.sparkContext.uiWebUrl)
except Exception as e:
    print("Ожидаемо — SparkContext недоступен:", type(e).__name__)

SparkSession остановлена
Ожидаемо — SparkContext недоступен: AttributeError
